In [3]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/orders.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [4]:
file_path = Path(FILE_PATH)

if not file_path.is_absolute():
    candidates = [Path.cwd() / file_path] + [
        parent / file_path for parent in Path.cwd().parents
    ]
    file_path = next((path for path in candidates if path.exists()), file_path)

if not file_path.exists():
    raise FileNotFoundError(f"Dataset not found: {FILE_PATH}")

df = pd.read_csv(file_path)  # Load the raw dataset before making changes.
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.


Loaded orders.csv
Rows: 45,000
Columns: 12


,order_id,customer_id,order_date,channel,store_id,fulfillment_method,order_status,payment_method,gross_order_value,discount_amount,shipping_fee,tax_amount
0,ORD-00000001,CUS-006832,2025-06-28,Website,NaN,Home Delivery,Completed,Digital Wallet,441.55,31.06,69.0,61.57
1,ORD-00000002,CUS-002041,2023-12-25,Website,NaN,Home Delivery,Completed,Digital Wallet,163.90,0.00,49.0,24.58
2,ORD-00000003,CUS-006564,2024-05-10,Mobile,NaN,Home Delivery,Completed,Cash,235.46,11.85,69.0,33.54
3,ORD-00000004,CUS-002664,2021-08-12,Store,STR-034,In Store,Completed,Buy Now Pay Later,676.07,14.54,0.0,99.23
4,ORD-00000005,CUS-000822,2020-09-16,Store,STR-039,In Store,Completed,Card,157.93,0.00,0.0,23.69


In [5]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,45000
1,columns,12
2,duplicates,0
3,missing_cells,25855


Decision point: determine which findings require remediation.


In [6]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,order_id,str,45000,0,45000
1,customer_id,str,43461,1539,11699
2,order_date,str,45000,0,2434
3,channel,str,45000,0,8
4,store_id,str,20684,24316,50
5,fulfillment_method,str,45000,0,7
6,order_status,str,45000,0,3
7,payment_method,str,45000,0,5
8,gross_order_value,float64,45000,0,21585
9,discount_amount,float64,45000,0,4839


In [7]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count
store_id,24316
customer_id,1539


In [8]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [9]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


,count,mean,std,min,25%,50%,75%,max
gross_order_value,45000.0,260.000125,237.945717,9.72,99.35,191.72,342.7000,3255.34
discount_amount,45000.0,9.499436,21.215801,0.00,0.00,0.00,10.8300,492.96
shipping_fee,45000.0,23.754089,34.132174,0.00,0.00,0.00,49.0000,89.00
tax_amount,45000.0,37.574990,34.497606,1.24,14.23,27.67,49.4925,488.30


,iqr_extreme_rate
discount_amount,0.110000
tax_amount,0.051133
gross_order_value,0.050711
shipping_fee,0.000000


In [10]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,order_id,45000,0,"{'ORD-00000001': 1, 'ORD-00000002': 1, 'ORD-00..."
1,customer_id,11699,0,"{nan: 1539, 'CUS-007859': 13, 'CUS-001571': 13..."
2,order_date,2434,0,"{'2026-02-14': 33, '2024-05-24': 32, '2022-11-..."
3,channel,8,0,"{'Store': 20522, 'Website': 12894, 'Mobile': 7..."
4,store_id,50,0,"{nan: 24316, 'STR-020': 464, 'STR-023': 452, '..."
5,fulfillment_method,7,0,"{'In Store': 20590, 'Home Delivery': 14545, 'C..."
6,order_status,3,0,"{'Completed': 40491, 'Cancelled': 2678, 'Pendi..."
7,payment_method,5,0,"{'Card': 9116, 'Digital Wallet': 9026, 'Cash':..."


In [11]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


,column,parse_failures,min,max
0,order_date,0,2020-01-01,2026-08-30


In [12]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,order_id,1.000
8,gross_order_value,0.480
1,customer_id,0.260
11,tax_amount,0.231
9,discount_amount,0.108
2,order_date,0.054
4,store_id,0.001
3,channel,0.000
5,fulfillment_method,0.000
6,order_status,0.000


In [13]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if numeric_pairs != []: display(numeric_pairs.sort_values("correlation",key=lambda s:s.abs(),ascending=False).head(20))  # Inspect strongest observed numeric relationships.


ValueError: Unable to coerce to Series, length must be 3: given 0

In [ ]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


In [ ]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
audit_summary.to_csv(f"outputs/{stem}_audit_summary.csv",index=False)  # Save the audit summary for downstream review.
